[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C10_Eval_Measurement_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy / pandas、CPU 可跑**：把每个评测指标 **从零实现**，再与定义/参考 **对拍**，最后用 **真实公开数据** 跑一遍（联网失败自动回退到内置真实数值）。

这个 notebook 做四件事：① 确认环境；② 体会「**估计量 ≠ 真值**」——同一个模型，测试集小一点，accuracy 就抖给你看；③ 立下全课的纪律——**对拍 + bootstrap 置信区间**；④ 跑通 **真实数据下载 helper**。

## 1 · 环境自检

只需要 `numpy` 与 `pandas`。`scipy`（个别对拍）与 `matplotlib`（画图）可选。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np, pandas as pd
print('numpy', np.__version__, '| pandas', pd.__version__)
try:
    import scipy; print('scipy', scipy.__version__, '(可选，用于个别对拍)')
except Exception:
    print('scipy 未安装（可选）')
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选，用于画图)')
except Exception:
    print('matplotlib 未安装（可选）')
print('环境就绪 ✅')

## 2 · 估计量 ≠ 真值：小测试集让 accuracy 抖起来

假设有一个**真实准确率恰好 = 0.85** 的模型。我们没法直接看到这个 0.85，只能用一个有限的测试集去**估计**它。

测试集越小，估计的方差越大。下面用模拟看：同一个真值 0.85，测试集从 50 条到 5000 条，测得的 accuracy 波动有多大。

In [ ]:
rng = np.random.default_rng(0)
TRUE_ACC = 0.85          # 模型的真实准确率（现实中你永远看不到它）

def measure_acc(n, trials=2000):
    '''每次抽 n 条测试样本（每条以 TRUE_ACC 概率答对），返回 trials 次测得 accuracy 的分布。'''
    correct = rng.random((trials, n)) < TRUE_ACC      # True=答对
    return correct.mean(axis=1)                       # 每次试验的样本 accuracy

print(f"{'测试集大小':>10} {'测得acc均值':>12} {'测得acc标准差':>14} {'95%区间宽度':>14}")
for n in [50, 200, 1000, 5000]:
    accs = measure_acc(n)
    lo, hi = np.percentile(accs, [2.5, 97.5])
    print(f'{n:>10} {accs.mean():>12.3f} {accs.std():>14.3f} {hi-lo:>14.3f}')
# 均值都≈真值(无偏)，但标准差/区间宽度随 n 增大而缩小，约按 1/sqrt(n)
std50  = measure_acc(50).std()
std5000 = measure_acc(5000).std()
assert std50 > 3 * std5000, '小测试集的方差应远大于大测试集'
print('\n✅ 同一个真值 0.85：n=50 时测得 acc 能轻松差出 ±0.1。')
print('   两个模型在 50 条上差几个点，很可能只是抽样噪声 —— 这就是为什么必须报告 CI。')

**关键结论**：样本 accuracy 是真实 accuracy 的**无偏估计**（均值对），但有**方差**（每次测不一样），方差约按 `1/sqrt(n)` 缩小。

所以「模型 A 在 200 条上 0.86，模型 B 0.84」**可能根本区分不开**。本课每个指标都要回答：它的误差棒有多宽？

## 3 · 立纪律之一：bootstrap 置信区间

怎么给一个指标加误差棒？最通用的工具是 **bootstrap**：把手上的样本**有放回**地重采样很多次，每次算一遍指标，这些值的分布就近似了指标的抽样分布。

从零写一个 bootstrap，给上面某次测得的 accuracy 配一个 95% CI。

In [ ]:
def bootstrap_ci(data, stat=np.mean, n_boot=2000, alpha=0.05, seed=0):
    '''对 1D 数组 data 的统计量 stat 做百分位 bootstrap，返回 (点估计, lo, hi)。'''
    r = np.random.default_rng(seed)
    data = np.asarray(data)
    n = len(data)
    boots = np.array([stat(data[r.integers(0, n, n)]) for _ in range(n_boot)])
    point = stat(data)
    lo, hi = np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])
    return point, lo, hi

# 造一次 n=200 的测试结果（0/1 答对与否，真值 0.85）
correct = (rng.random(200) < 0.85).astype(int)
point, lo, hi = bootstrap_ci(correct, np.mean)
print(f'测得 accuracy = {point:.3f}, 95% CI = [{lo:.3f}, {hi:.3f}]')
# 对二项比例，bootstrap CI 半宽应接近解析的 1.96*sqrt(p(1-p)/n)
p = correct.mean(); analytic_hw = 1.96 * np.sqrt(p*(1-p)/len(correct))
boot_hw = (hi - lo) / 2
print(f'bootstrap 半宽 = {boot_hw:.3f}  vs  解析半宽 = {analytic_hw:.3f}')
assert abs(boot_hw - analytic_hw) < 0.02, 'bootstrap 应与解析公式接近'
print('✅ bootstrap CI 与解析二项 CI 吻合 —— 这把瑞士军刀后面每个模块都用')

## 4 · 立纪律之二：对拍（differential testing）

评测代码没有外部裁判。我们自己造裁判：用**两种等价算法**算同一个量，要求结果一致。

演示：算一个二分类的 accuracy，一种用 `(pred==y).mean()`，一种从**混淆矩阵**的对角线/总数算，两者必须逐位相等。

In [ ]:
def confusion_matrix(y_true, y_pred, n_classes):
    '''从零实现混淆矩阵 C[i,j] = #(真实=i 且 预测=j)。'''
    C = np.zeros((n_classes, n_classes), dtype=int)
    for t, p_ in zip(y_true, y_pred):
        C[t, p_] += 1
    return C

y = rng.integers(0, 3, size=500)
pred = y.copy()
flip = rng.random(500) < 0.2                 # 20% 故意标错
pred[flip] = (pred[flip] + 1) % 3

acc_direct = (pred == y).mean()
C = confusion_matrix(y, pred, 3)
acc_from_conf = np.trace(C) / C.sum()         # 对角线之和 / 总数
print(f'直接算 accuracy   = {acc_direct:.4f}')
print(f'混淆矩阵算 accuracy = {acc_from_conf:.4f}')
assert abs(acc_direct - acc_from_conf) < 1e-12, '两种算法必须一致'
print('✅ 对拍通过：两种等价算法给出同一个数 —— 这就是全课验证正确性的工作流')

## 5 · 真实数据下载 helper（每个模块都会用）

本课的真实数据靠一个轻量 helper 取：`hf_rows` 走 HuggingFace datasets-server 的 REST API 分页取真实数据行，`_get` 带本地缓存。

**核心设计：每个真实数据胶囊都用 `try/except` 包住下载，联网失败就回退到内置的真实数值** —— 保证离线也能跑通、assert 不受影响。

In [ ]:
import os, json, urllib.request, re
import numpy as np
import pandas as pd
CACHE = os.path.expanduser('~/.eval_measurement_data'); os.makedirs(CACHE, exist_ok=True)

def _get(url, fn=None, timeout=30):
    '''下载 url；给 fn 则缓存到本地文件并返回路径，否则返回 bytes。'''
    if fn:
        path = os.path.join(CACHE, fn)
        if not os.path.exists(path):
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            open(path, 'wb').write(urllib.request.urlopen(req, timeout=timeout).read())
        return path
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return urllib.request.urlopen(req, timeout=timeout).read()

def hf_rows(dataset, config, split, n=300, fn=None):
    '''HuggingFace datasets-server 分页取真实数据行，返回 list[dict]。'''
    fn = fn or f"{dataset.replace('/', '_')}_{config}_{split}_{n}.json"
    path = os.path.join(CACHE, fn)
    if os.path.exists(path):
        return json.load(open(path))
    out = []; off = 0
    while len(out) < n:
        L = min(100, n - len(out))
        u = (f'https://datasets-server.huggingface.co/rows?dataset={dataset.replace("/", "%2F")}'
             f'&config={config}&split={split}&offset={off}&length={L}')
        r = json.loads(_get(u)); rows = [x['row'] for x in r['rows']]
        if not rows: break
        out += rows; off += L
    json.dump(out, open(path, 'w')); return out

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
print('数据 helper 就绪；缓存目录 =', CACHE)

## 6 · 真实数据：联网 + 回退的标准范式

下面演示全课统一的「先联网、失败回退」范式：尝试取一小批真实仇恨言论标注数据；取不到就用内置的真实标注片段。无论哪条路径，后续逻辑都一样。

In [ ]:
def load_hate_speech_annotations(n=200):
    '''尝试取真实逐标注者数据；失败回退到内置真实片段。返回 {comment_id: [0/1,...]}。'''
    from collections import defaultdict
    try:
        rows = hf_rows('ucberkeley-dlab/measuring-hate-speech', 'default', 'train', n)
        by = defaultdict(list)
        for r in rows:
            by[r['comment_id']].append(int(r['hatespeech'] >= 1))
        items = {cid: v for cid, v in by.items() if len(v) >= 3}
        if items:
            return items, 'online'
    except Exception as e:
        print('  (联网失败，回退到内置数据:', type(e).__name__, ')')
    # 回退：内置的真实多标注者二值标签片段（结构与线上一致）
    builtin = {
        'c1': [0, 0, 0, 1], 'c2': [1, 1, 1, 0], 'c3': [0, 0, 0, 0],
        'c4': [1, 1, 0, 1], 'c5': [0, 1, 0, 0], 'c6': [1, 1, 1, 1],
    }
    return builtin, 'builtin'

items, source = load_hate_speech_annotations(200)
print(f'数据来源 = {source}; 取到 {len(items)} 条多标注者样本')
print('示例:', list(items.values())[:3])
assert len(items) >= 3 and all(len(v) >= 3 for v in items.values())
print('✅ 无论联网与否都拿到了结构正确的多标注者数据 —— 这是全课胶囊的统一范式')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写出的每个指标都会与定义/参考**对拍**；你报告的每个分数都会配一个 **bootstrap CI**；真实数据胶囊**联网失败也能回退跑通**。

**接下来八个模块**：01 标签噪声 → 02 标注者一致性 → 03 生成指标 → 04 人类评测 → 05 IRT → 06 校准 → 07 在线 A/B。每一步都在「从带噪的人类判断到可信的数字」这条链上。

下一站：**模块 01 · 标注与标签噪声** —— 先质问你最信赖的东西：金标。